# Module 1 — Object-Oriented Programming in Python
**Runnable companion notebook.** Run each cell (`Shift+Enter`) and read the output. Sections: classes & objects → encapsulation → inheritance → polymorphism/dunders → abstraction → class vs static methods → a mini-project → practice.

> OOP bundles **data + behaviour** into objects that model real-world things.

## 1. Classes & objects
A **class** is a blueprint; an **object** is an instance. `__init__` sets up state; `self` is the current object.

In [1]:
class Student:
    school = 'ABC'                 # class attribute (shared by all)
    def __init__(self, name, cgpa):
        self.name = name           # instance attributes (per object)
        self.cgpa = cgpa
    def greet(self):               # instance method
        return f'Hi, I am {self.name} (CGPA {self.cgpa}) from {self.school}'

s1 = Student('Ravi', 8.4)
s2 = Student('Mira', 9.1)
print(s1.greet())
print(s2.greet())
print('Same blueprint, different state:', s1.cgpa, 'vs', s2.cgpa)

Hi, I am Ravi (CGPA 8.4) from ABC
Hi, I am Mira (CGPA 9.1) from ABC
Same blueprint, different state: 8.4 vs 9.1


## 2. Encapsulation
Hide internal state behind a clean interface. A leading `__` makes an attribute *private* (name-mangled). `@property` gives a read-only view; methods validate changes.

In [2]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.__balance = balance        # private
    @property
    def balance(self):                  # read-only getter
        return self.__balance
    def deposit(self, amt):
        if amt <= 0:
            raise ValueError('deposit must be positive')
        self.__balance += amt
    def withdraw(self, amt):
        if amt > self.__balance:
            raise ValueError('insufficient funds')
        self.__balance -= amt

acc = BankAccount('Asha', 1000)
acc.deposit(500)
acc.withdraw(200)
print('Balance:', acc.balance)

# the interface protects the object from invalid states:
try:
    acc.deposit(-50)
except ValueError as e:
    print('Blocked:', e)

Balance: 1300
Blocked: deposit must be positive


## 3. Inheritance
A child class reuses a parent and overrides behaviour. `super()` calls the parent.

In [3]:
class Animal:
    def __init__(self, name):
        self.name = name
    def speak(self):
        return '...'

class Dog(Animal):
    def speak(self):           # override
        return 'Woof'

class Cat(Animal):
    def speak(self):
        return 'Meow'

class Puppy(Dog):              # multilevel: Puppy -> Dog -> Animal
    def __init__(self, name):
        super().__init__(name)
    def speak(self):
        return 'Yip'

for a in [Dog('Rex'), Cat('Tom'), Puppy('Bz')]:
    print(f'{a.name:4s} -> {a.speak()}')

Rex  -> Woof
Tom  -> Meow
Bz   -> Yip


## 4. Polymorphism & dunder methods
Same call, different behaviour. Dunder methods make your objects work with `print`, `+`, `==`, `len`, etc.

In [4]:
class Vector2D:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __repr__(self):
        return f'Vector2D({self.x}, {self.y})'
    def __add__(self, other):
        return Vector2D(self.x + other.x, self.y + other.y)
    def __eq__(self, other):
        return (self.x, self.y) == (other.x, other.y)

a = Vector2D(1, 2)
b = Vector2D(3, 4)
print('a + b =', a + b)        # uses __add__ and __repr__
print('a == Vector2D(1,2):', a == Vector2D(1, 2))

a + b = Vector2D(4, 6)
a == Vector2D(1,2): True


## 5. Abstraction (abstract base classes)
Define *what* subclasses must do; they fill in *how*. An abstract class can't be instantiated directly.

In [5]:
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @abstractmethod
    def area(self):
        ...

class Circle(Shape):
    def __init__(self, r): self.r = r
    def area(self): return math.pi * self.r**2

class Rectangle(Shape):
    def __init__(self, w, h): self.w, self.h = w, h
    def area(self): return self.w * self.h

shapes = [Circle(2), Rectangle(3, 4)]
for sh in shapes:
    print(type(sh).__name__, 'area =', round(sh.area(), 2))

try:
    Shape()                    # cannot instantiate an abstract class
except TypeError as e:
    print('Blocked:', e)

Circle area = 12.57
Rectangle area = 12
Blocked: Can't instantiate abstract class Shape without an implementation for abstract method 'area'


## 6. Instance vs class vs static methods

In [6]:
class Pizza:
    count = 0                         # class attribute
    def __init__(self, size):
        self.size = size
        Pizza.count += 1
    def describe(self):               # instance method (needs self)
        return f'a {self.size} pizza'
    @classmethod
    def total_made(cls):              # works on the class
        return cls.count
    @staticmethod
    def is_valid(size):               # utility, no self/cls
        return size in ('S', 'M', 'L')

p1 = Pizza('M'); p2 = Pizza('L')
print(p1.describe())
print('valid size XL?', Pizza.is_valid('XL'))
print('pizzas made:', Pizza.total_made())

a M pizza
valid size XL? False
pizzas made: 2


## 7. Mini-project — a tiny library system
Puts the pillars together: encapsulation (private list), behaviour (borrow/return), and a readable `__repr__`.

In [7]:
class Book:
    def __init__(self, title, copies=1):
        self.title = title
        self.copies = copies
    def __repr__(self):
        return f'{self.title} ({self.copies} left)'

class Library:
    def __init__(self):
        self.__catalog = {}
    def add(self, book):
        self.__catalog[book.title] = book
    def borrow(self, title):
        b = self.__catalog.get(title)
        if not b or b.copies == 0:
            return f'"{title}" not available'
        b.copies -= 1
        return f'You borrowed "{title}"'
    def __repr__(self):
        return 'Library: ' + ', '.join(str(b) for b in self.__catalog.values())

lib = Library()
lib.add(Book('Python 101', 2))
lib.add(Book('Algorithms', 1))
print(lib.borrow('Algorithms'))
print(lib.borrow('Algorithms'))   # now 0 left
print(lib)

You borrowed "Algorithms"
"Algorithms" not available
Library: Python 101 (2 left), Algorithms (0 left)


## 8. Practice exercises
Try these yourself in the empty cells below.

1. Create a `Rectangle` class with `width`, `height`, an `area()` method and a `__str__`.
2. Add a `Square` subclass that reuses `Rectangle` via `super().__init__`.
3. Extend `BankAccount` with a transaction history (a private list) and a `statement()` method.
4. Add `__lt__` to `Vector2D` so vectors sort by magnitude.
5. Add an alternative constructor `Student.from_csv_row('Ravi,8.4')` using `@classmethod`.


In [8]:
# Worked sample for #1–#2 (study it, then attempt the rest):
class Rectangle:
    def __init__(self, width, height):
        self.width, self.height = width, height
    def area(self):
        return self.width * self.height
    def __str__(self):
        return f'Rectangle {self.width}x{self.height} (area={self.area()})'

class Square(Rectangle):
    def __init__(self, side):
        super().__init__(side, side)

print(Rectangle(3, 4))
print(Square(5))

Rectangle 3x4 (area=12)
Rectangle 5x5 (area=25)


In [9]:
# Your turn: try exercises 3, 4, 5 here
